# TÜV Thüringen control notebook

Store group *Prüfanlagen* (Shopware + Acris Store Locator, server-rendered). **Live requests only with `CONFIRM_LIVE = True`**: main page, cross-check URL, then the Service-Center / Kunden-Center scan (4 requests, 2–4 s apart). Offline it works on the newest dev copy in `test_output/*_sample/page.html`.

In [ ]:
from datetime import datetime
from IPython.display import display
import pandas as pd
from core import paths
from tuev_thueringen import adapter as tt

CONFIRM_LIVE = False  # True = live requests + official run in output/; False = offline on the dev page, writes to test_output/
STARTED = datetime.now()
MODE = "full" if CONFIRM_LIVE else "recon"
pd.set_option("display.width", 200, "display.max_colwidth", 40)
print("live" if CONFIRM_LIVE else "offline", MODE, f"{STARTED:%Y-%m-%d %H:%M}")

In [ ]:
if CONFIRM_LIVE:
    main_html = tt.fetch(tt.MAIN_URL)
    cross_html = tt.fetch(tt.CROSSCHECK_URL)
    print("live: main", len(main_html), "chars, cross-check", len(cross_html), "chars")
else:
    dev_page = sorted(paths.scraper_dir(tt.SLUG).glob("test_output/*_sample/page.html"))[-1]
    main_html, cross_html = dev_page.read_text(encoding="utf-8"), None
    print("offline:", dev_page.relative_to(paths.project_root()), len(main_html), "chars (no cross-check)")

In [ ]:
df, ids = tt.build(main_html, scraped_at=STARTED.isoformat(timespec="seconds"))
print(len(df), "stores;", "ids per view:", {k: len(v) for k, v in ids.items()})
print("needs_review:", int(df["needs_review"].sum()), "| flags:", df["doubt_flags"].str.split(";").explode().replace("", pd.NA).value_counts().to_dict())
display(df[["city", "zip", "name", "email", "website", "plz_distance_km", "doubt_flags"]].head(5))

In [ ]:
cross_ids = tt.parse(cross_html)[1]["table"] if cross_html else None
checks = tt.check(df, ids, cross_ids, tt.last_pass_count())
for name, ok, detail in checks:
    print("PASS" if ok else "FAIL", "-", name, "-", detail)
STATUS = tt.verdict(checks)
print("\nOVERALL:", STATUS)
print("review flags:"); display(df.loc[df["needs_review"], ["city", "needs_review_reason"]])

In [ ]:
csv_path = tt.export_run(df, STATUS, MODE, STARTED)
print(STATUS, "->", csv_path.relative_to(paths.project_root()), len(df), "rows")

In [ ]:
# Report-only: nothing is written, nothing is added to the CSV.
if not CONFIRM_LIVE:
    print("skipped (offline); set CONFIRM_LIVE = True to fetch the two groups")
else:
    for group, url in tt.SCAN_URLS.items():
        hits = tt.scan_group(tt.fetch(url))
        print(f"{group}: {len(hits)} entries mention HU / Prüfanlage / Prüfstelle")
        for h in hits:
            print("  -", h["name"][:60], h["terms"], "|", h["text"][:100])